# Building Energy Efficiency Classification (Tutorial)

**Ungraded Tutorial** - Classify buildings as energy-efficient or inefficient using PyTorch neural networks on aerial image embeddings from ~6,000 Rotterdam buildings.

## Learning Objectives
- Implement neural networks via PyTorch
- Instantiate neural networks
- Solve supervised learning (building energy efficiency classification)
- (Optional) Implement a custom Dataset class

## 0. Load Modules

In [1]:
import gzip
import pickle
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

## 1. Torch Dataset

Load embeddings and labels, then create a Buildings dataset class.

In [2]:
path_to_embeddings = "../data/beec_dataset/embeddings/"

with open(path_to_embeddings + "embeddings_train.pkl.gz", "rb") as f:
    embeddings_train = pickle.loads(gzip.decompress(f.read()))

with open(path_to_embeddings + "embeddings_test.pkl.gz", "rb") as f:
    embeddings_test = pickle.loads(gzip.decompress(f.read()))

print(embeddings_train.shape)
print(embeddings_test.shape)

torch.Size([4499, 384])
torch.Size([1500, 384])


In [3]:
path_to_labels = "../data/beec_dataset/labels/"

df_train = pd.read_csv(path_to_labels + "train.csv")
df_test = pd.read_csv(path_to_labels + "test.csv")

train_labels = torch.tensor(df_train["label"].tolist(), dtype=torch.long)
test_labels = torch.tensor(df_test["label"].tolist(), dtype=torch.long)

print(len(train_labels), len(test_labels))

4499 1500


In [4]:
class Buildings(Dataset):
    def __init__(self, input_tensor, output_tensor, transform=None):
        assert len(output_tensor) == len(input_tensor)
        self.input_tensor = input_tensor
        self.output_tensor = output_tensor
        self.transform = transform

    def __len__(self):
        return len(self.output_tensor)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        return self.input_tensor[idx], self.output_tensor[idx]

train_dataset = Buildings(embeddings_train, train_labels)
test_dataset = Buildings(embeddings_test, test_labels)
train_dataset[0]

(tensor([ 3.0472e-01, -1.0400e+00,  7.5045e-01,  9.4056e-01, -1.9510e+00,
         -1.3022e+00,  1.2784e-01, -3.1624e-01, -1.6801e-02, -8.5304e-01,
          8.7940e-01,  7.7779e-01,  6.6031e-02,  1.1272e+00,  4.6751e-01,
         -8.5929e-01,  3.6875e-01, -9.5888e-01,  8.7845e-01, -4.9926e-02,
         -1.8486e-01, -6.8093e-01,  1.2225e+00, -1.5453e-01, -4.2833e-01,
         -3.5213e-01,  5.3231e-01,  3.6544e-01,  4.1273e-01,  4.3082e-01,
          2.1416e+00, -4.0642e-01, -5.1224e-01, -8.1377e-01,  6.1598e-01,
          1.1290e+00, -1.1395e-01, -8.4016e-01, -8.2448e-01,  6.5059e-01,
          7.4325e-01,  5.4315e-01, -6.6551e-01,  2.3216e-01,  1.1669e-01,
          2.1869e-01,  8.7143e-01,  2.2360e-01,  6.7891e-01,  6.7579e-02,
          2.8912e-01,  6.3129e-01, -1.4572e+00, -3.1967e-01, -4.7037e-01,
         -6.3888e-01, -2.7514e-01,  1.4949e+00, -8.6583e-01,  9.6828e-01,
         -1.6829e+00, -3.3489e-01,  1.6275e-01,  5.8622e-01,  7.1123e-01,
          7.9335e-01, -3.4873e-01, -4.

## 2. Neural Net Definition

Three linear layers with ReLU activations. Input: 384-dim embeddings. Output: 2 classes.

- **Linear layer**: `nn.Linear(input_size, output_size)` - fully connected transformation
- **ReLU**: `nn.ReLU()` - outputs input if positive, else zero

In [5]:
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = NeuralNet(input_size=384, hidden_size=256, num_classes=2)
print(model)

NeuralNet(
  (fc1): Linear(in_features=384, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=2, bias=True)
  (relu): ReLU()
)


## 3. Training Loop

In [6]:
seed = 42
torch.manual_seed(seed)

num_epochs = 40
batch_size = 32
learning_rate = 0.0001

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [7]:
def train_neural_net():
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for embeds, labels in train_loader:
            outputs = model(embeds)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * embeds.size(0)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss:.4f}")

        model.eval()
        total = correct = tp = fp = fn = 0
        with torch.no_grad():
            for embeds, labels in test_loader:
                outputs = model(embeds)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                tp += ((predicted == 1) & (labels == 1)).sum().item()
                fp += ((predicted == 1) & (labels == 0)).sum().item()
                fn += ((predicted == 0) & (labels == 1)).sum().item()
        accuracy = correct / total
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        print(f"Epoch [{epoch+1}/{num_epochs}], Test Accuracy: {accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

train_neural_net()

Epoch [1/40], Training Loss: 0.6943
Epoch [1/40], Test Accuracy: 0.48, Precision: 0.50, Recall: 0.37, F1: 0.43


Epoch [2/40], Training Loss: 0.6776
Epoch [2/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.48, F1: 0.49


Epoch [3/40], Training Loss: 0.6574
Epoch [3/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.39, F1: 0.44


Epoch [4/40], Training Loss: 0.6263
Epoch [4/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.49, F1: 0.50


Epoch [5/40], Training Loss: 0.5848
Epoch [5/40], Test Accuracy: 0.48, Precision: 0.49, Recall: 0.35, F1: 0.41


...

Epoch [36/40], Training Loss: 0.0004
Epoch [36/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.49, F1: 0.50


Epoch [37/40], Training Loss: 0.0003
Epoch [37/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.48, F1: 0.49


Epoch [38/40], Training Loss: 0.0003
Epoch [38/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.48, F1: 0.50


Epoch [39/40], Training Loss: 0.0003
Epoch [39/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.48, F1: 0.50


Epoch [40/40], Training Loss: 0.0003
Epoch [40/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.48, F1: 0.50
